# SIMR tour
A hands-on tour of the SIMR workflow (`galactic_tidal_constraints.simr.simr`): it runs the simulations a request asks
for, for many models, and hands back only the data the request names. Run the cells in order.

`sim` (see `simulator_tour.ipynb`) runs one simulation. `simr` sits above it: it decides which simulations to run,
runs them and collects the results. So far it covers the Simulator stage only, keeps everything in memory, and runs
models one after another. Saving, failure handling, the config file and pools come in later steps.

In [ ]:
%matplotlib inline
import time

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

## Models
`simr` takes many models at once: an array of shape `(N, 14)`, one model per row, in `sim.PARAM_NAMES` order. A single
model of shape `(14,)` also works.

Here, line 3 of JSPAM's `archive/data.txt` and the same with the secondary's mass doubled. Its secondary passes inside
r = 2, where dynamical friction acts, so `lnl` changes its runs (see **Dynamical friction** in `simulator_tour.ipynb`;
a wider model gives identical runs for every `lnl`):

In [ ]:
params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
models = np.stack([params, heavier])
print("models:", models.shape)

## A request
A request is a plain dict, shaped like the config file to come:

| Key | Default | What it does |
|---|---|---|
| `run.return` | `[]` | Names of the data to hand back (below) |
| `run.save` | `[]` | Names of the data to save in `output` (see **Saving and loading**). Independent of `return` |
| `simulator` | one `SimulatorConfig()` | A dict (or `SimulatorConfig`), or a list of them. Each dict's keys are `SimulatorConfig` fields, `seed` included |

Every model runs on every simulator entry. The names the Simulator stage can hand back (`simr.SIM_DATA`):

| Name | Each cell holds |
|---|---|
| `sim.particles` | `sim.simulate_particles`: every particle at the end, `(n1 + n2 + 1, 6)`, float32 |
| `sim.unperturbed` | `sim.unperturbed_particles`: the same disks at today's centers, `(n1 + n2 + 1, 6)`, float32 |
| `sim.orbit` | `sim.simulate_orbit`: the secondary's center at every step, `(n_steps + 1, 7)`, float32 |
| `sim.timing` | the dict from `sim.get_timing` (start time, steps, which pass) |

In [ ]:
N = 1500
request = {
    "run": {"return": ["sim.particles", "sim.orbit", "sim.timing", "sim.unperturbed"]},
    "simulator": [{"n1": N, "n2": N, "lnl": 0.0025}, {"n1": N, "n2": N, "lnl": 0.001}],
}
result = simr.run(models, request)
print(list(result.data))

## The result
`simr.run` returns a `SimrResult`, which describes itself:
- `result.models`: the models that ran, `(N, 14)`: row m is model m.
- `result.simulators`: the simulator entries as `SimulatorConfig`s, with every default filled in (the seed too): entry s is simulator s.
- `result.data`: one entry per requested name, and nothing else.

Each entry is an array of cells indexed `[model, simulator]`. A cell holds that run's own array, so orbits of
different lengths fit side by side. Later, scores will sit beside `data` as `result.scores[model, simulator, image, fitness]`.

In [ ]:
print("models:", result.models.shape, " mass2 of each:", result.models[:, sim.PARAM_INDEX["mass2"]])
for s, config in enumerate(result.simulators):
    print(f"simulator {s}: lnl {config.lnl}, seed {config.seed}, n1 {config.n1}, heat_mode {config.heat_mode!r}, start {config.start!r}")
for name, cells in result.data.items():
    print(f"{name}: {cells.shape}")
for m in range(len(result.models)):
    for s in range(len(result.simulators)):
        orbit = result.data["sim.orbit"][m, s]
        timing = result.data["sim.timing"][m, s]
        print(f"  [{m}, {s}]  orbit {orbit.shape} {orbit.dtype}   t_start {timing['t_start']:.3f}   start {timing['start']}")

Each row is one model, each column one simulator entry. Friction changes each run's history (when it starts, and when
and how close the pass was), while every run still ends at the observed position:

In [ ]:
n_models, n_sims = len(result.models), len(result.simulators)
fig, axes = plt.subplots(n_models, n_sims, figsize=(6 * n_sims, 6 * n_models), sharex=True, sharey=True, squeeze=False)
for m, model in enumerate(result.models):
    for s, config in enumerate(result.simulators):
        ax, p, orbit = axes[m, s], result.data["sim.particles"][m, s], result.data["sim.orbit"][m, s]
        n1 = config.n1
        ax.scatter(p[:n1, 0], p[:n1, 1], s=1, label="primary disk")
        ax.scatter(p[n1:-1, 0], p[n1:-1, 1], s=1, label="secondary disk")
        ax.plot(orbit[:, 0], orbit[:, 1], color="k", lw=1, label="secondary's orbit")
        mass2 = model[sim.PARAM_INDEX["mass2"]]
        ax.set_title(f"[{m}, {s}]  mass2 {mass2:.3g}, lnl {config.lnl:g}: starts at t = {orbit[0, 6]:.2f}")
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.5, 2.5)
        ax.set_aspect("equal")
axes[0, 0].legend(loc="upper left", markerscale=5)
plt.show()

## Only what's requested runs
Asking only for the orbit or the timing never runs particles:

In [ ]:
for names in (["sim.orbit"], ["sim.timing"], ["sim.unperturbed"], ["sim.particles"]):
    start = time.perf_counter()
    simr.run(models, {**request, "run": {"return": names}})
    print(f"{str(names):22s} {time.perf_counter() - start:6.2f} s for {len(models)} models x 2 entries")

## Saving and loading
`run.save` names the data to write, from the same list as `return`, and `simr.run(..., output=folder)` says where.
Each model gets a folder:

```
<output>/
    index.json                 "models": id -> 14 parameters, every model ever saved here;
                               "runs": one record per run that saved here (configs, model ids, times, code)
    model_000000/              one folder per model id
        model.json             14 named parameters, and each saved simulation by name ("sim_00" for entry 0):
                               config (seed included), timing, files, and the code it was made with
        simulations/
            sim_00_particles.npy   one file per saved array: <simulation>_<item>.npy, float32
            sim_00_orbit.npy
```

A run's models are matched exactly against the index: a model already there uses its folder, a new one gets the next
id at the bottom, and `result.ids` gives each row's id. So reordered or overlapping model lists land in the same
folders. One run at a time per output folder.

Each `.npy` is written atomically, and only then does `model.json` list it, so an array counts as saved only if
`model.json` lists it. With an output folder, a simulation saved under the same name with the same config is loaded
instead of computed, and gains any arrays it's missing; a different config is neither used nor changed. `simr.load_model(folder)` and
`simr.load_simulation(folder, name)` read them back exactly as `run` returns them.

In [ ]:
import tempfile
from pathlib import Path

output = Path(tempfile.mkdtemp(prefix="simr_tour_"))
save_request = {**request, "run": {"save": ["sim.particles", "sim.orbit"]}}
saved = simr.run(models, save_request, output=output)
print("ids:", saved.ids)
for path in sorted(output.rglob("*")):
    print(path.relative_to(output))

index = simr.load_index(output)
print("index models:", list(index["models"]), " runs:", len(index["runs"]))
record = simr.load_model(output / "model_000001")
print("\nmodel_000001 mass2:", record["model"]["mass2"])
for name, entry in record["simulations"].items():
    print(f"{name}: lnl {entry['config']['lnl']}, seed {entry['config']['seed']}, t_start {entry['timing']['t_start']:.2f}, "
          f"files {list(entry['files'])}, jspam {entry['code']['jspam']['commit'][:7]}")

loaded = simr.load_simulation(output / "model_000001", "sim_00")
print("\nloaded:", {name: (array.shape, str(array.dtype)) for name, array in loaded.items()})
print("same as returned:", np.array_equal(loaded["sim.particles"], result.data["sim.particles"][1, 0]))

## Mistakes in a request
Unknown keys, names and fields, and bad values, are caught before any simulation runs, and the message says where:

In [ ]:
bad_requests = [
    {"runn": {}},
    {"run": {"return": ["sim.orbits"]}},
    {"run": {"return": "sim.orbit"}},
    {"run": {"seed": 3}},
    {"simulator": {"seed": 1.5}},
    {"simulator": [{"n1": 10}, {"n_1": 10}]},
    {"simulator": [{"lnl": -1}]},
]
for bad in bad_requests:
    try:
        simr.run(models, bad)
    except ValueError as error:
        print(f"{str(bad):42s} -> {error}")

## Plots from each step
The plots made while each step was built, collected here. Each subsection runs on its own after the setup cells at the
top. Collapse a subsection by clicking the bar to the left of its heading.

### Step 8a · Requests and returned data
The plots for this step are above, under **The result**: two models on two `lnl` values from one request. Both entries
have seed 0, so each row's two runs start from the same disks.

### Step 8b · Saving and loading
Both models on both entries, read back from disk (saved under **Saving and loading** above), with no simulation run:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharex=True, sharey=True)
for m, folder in enumerate(sorted(output.glob("model_*"))):
    record = simr.load_model(folder)
    for s, (name, entry) in enumerate(record["simulations"].items()):
        loaded = simr.load_simulation(folder, name)
        p, orbit, n1 = loaded["sim.particles"], loaded["sim.orbit"], entry["config"]["n1"]
        axes[m, s].scatter(p[:n1, 0], p[:n1, 1], s=1)
        axes[m, s].scatter(p[n1:-1, 0], p[n1:-1, 1], s=1)
        axes[m, s].plot(orbit[:, 0], orbit[:, 1], color="k", lw=1)
        axes[m, s].set_title(f"{folder.name}/{name}: mass2 {record['model']['mass2']:.3g}, lnl {entry['config']['lnl']:g}")
        axes[m, s].set_xlim(-2.5, 2.5)
        axes[m, s].set_ylim(-2.5, 2.5)
        axes[m, s].set_aspect("equal")
plt.show()